In [10]:
!pip uninstall -y numpy
!pip install "numpy<2.0"

!pip install torch==2.2.2 torchvision==0.17.2 monai
!pip install scikit-image
!pip install wandb
!pip install import-ipynb
!pip install nibabel 


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/lib/python3.10/dist-packages/numpy-1.26.4.dist-info/' -> '/tmp/pip-uninstall-3w345l3z'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 105, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", line 675, in uninstall
    uninstalle

In [11]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
import monai
import torch
import re
import torchvision.transforms as transforms
from PIL import Image
from monai.transforms import LoadImage
import import_ipynb
from Functions import patients_dicts
zip_file = "Resources.zip"
os.makedirs("train", exist_ok=True)
!unzip -o -q {zip_file} -d {"train"}
data_path = "train"

train_dict_list = patients_dicts(data_path)

In [13]:
class MedMNISTData(monai.data.Dataset):
    
    def __init__(self, datafile, transform=None):
        self.data = datafile
        self.transform = transform
        
        
    def __getitem__(self, index):
        transfrom_dict = {
            "imgED": self.data[index]['imgED'].get_fdata(),
            "maskED": self.data[index]['maskED'].get_fdata(),
            "imgES": self.data[index]['imgES'].get_fdata(),
            "maskES": self.data[index]['maskES'].get_fdata(),
            "ID": self.data[index].get('ID', None),
            "disease": self.data[index].get('Disease', None),
            "imgED_header": self.data[index]['imgED'].header,
            "maskED_header": self.data[index]['maskED'].header,
            "imgES_header": self.data[index]['imgES'].header,
            "maskES_header": self.data[index]['maskES'].header}

        if self.transform:
            transfrom_dict = self.transform(transfrom_dict)
        return transfrom_dict
    
    def __len__(self):
        return len(self.data)

In [14]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd,ScaleIntensityd
#from monai.data import Dataset

data_transform = Compose([
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel"),
    ScaleIntensityd(keys=["imgED", "imgES", ])
])

train_dataset = MedMNISTData(train_dict_list, transform=data_transform)

In [15]:
sizes = []

for patient in range(len(train_dataset)):
    shapeED = train_dataset[patient]['maskED'].shape
    shapeES = train_dataset[patient]['maskES'].shape
    sizes.append(shapeED)
    sizes.append(shapeES)

maxx = 0; maxy = 0; maxz = 0
for images in range(len(sizes)):
    maxx = max(maxx,sizes[images][1])
    maxy = max(maxy,sizes[images][2])
    maxz = max(maxz,sizes[images][3])
    max_size = [1, maxx, maxy, maxz]

print(max_size)


[1, 256, 256, 10]


In [16]:
def visualize_heart_sample(sample, title=None):
    # Visualize the x-ray and overlay the mask, using the dictionary as input
    for i in range(2):
        if i == 0:
            image = np.squeeze(sample['imgED'])
            mask = np.squeeze(sample['maskED'])
        else:
            image = np.squeeze(sample['imgES'])
            mask = np.squeeze(sample['maskES'])

        plt.figure(figsize=[10,7])
        plt.imshow(image, 'gray')

        mask1 = np.ma.masked_where(mask != 1, mask)
        plt.imshow(mask1, 'Greens', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask2 = np.ma.masked_where(mask != 2, mask)
        plt.imshow(mask2, 'Reds', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask3 = np.ma.masked_where(mask != 3, mask)
        plt.imshow(mask3, 'Blues', alpha = 0.5, clim=[0,1], interpolation='nearest')
        if title is not None:
            plt.title(title)
        plt.show()

for i in range(shape[2]):
    single_data = train_dataset[0]
    sample_slice = {
    'imgED': single_data['imgED'][0,:, :, i],
    'maskED': single_data['maskED'][0,:, :, i],
    'imgES': single_data['imgES'][0,:, :, i],
    'maskES': single_data['maskES'][0,:, :, i]
    }
    visualize_heart_sample(sample_slice, title=f"patient:{single_data['ID']},disease:{single_data['disease']},slice:{i}")


NameError: name 'shape' is not defined